<a href="https://colab.research.google.com/github/JasonL888/AI_Experiments/blob/main/VideoTranscribe/video_transcribe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Video Transcribe
Sample code that
- extracts the audio from a video file
- uses OpenAI whisper model to transcribe
    - auto speech recognition (w/o translation)
- writes a subtitle file
    - in original language

## Intall the pre-requisites

In [2]:
# Install the core libraries (PyTorch is often required by transformers)
!pip install torch transformers datasets soundfile
# Install moviepy for audio extraction (requires ffmpeg to be installed on your system)
!pip install moviepy imageio-ffmpeg

## Import modules

In [6]:
import os
import sys
from dotenv import load_dotenv

from transformers import pipeline
import torch

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
  from moviepy.editor import VideoFileClip
else:
  from moviepy import VideoFileClip

## Configuration

In [7]:
ASR_MODEL_NAME = "openai/whisper-base"
SAMPLE_VIDEO_FILEPATH = "sample_video.mp4"
SAMPLE_AUDIO_FILEPATH = "extracted_audio.mp3"
OUTPUT_SUBTITLE_FILEPATH = "sample_video.srt"

In [8]:
# Determine the GPU device if available
if torch.backends.mps.is_available():
    device = "mps"
    device_type = "Apple Silicon (MPS) GPU"
elif torch.cuda.is_available():
    device = 0 # Use first CUDA GPU
    device_type = "NVIDIA (CUDA) GPU"
else:
    device = -1 # Use CPU
    device_type = "CPU"

print(f"Targeting device: {device_type} ({device})")

Targeting device: NVIDIA (CUDA) GPU (0)


## Helper Functions

In [9]:
def format_time(seconds):
    """Converts seconds into the SRT time format (HH:MM:SS,mmm)."""
    if seconds is None:
        return "00:00:00,000"

    total_milliseconds = int(round(seconds * 1000))
    milliseconds = total_milliseconds % 1000
    total_seconds = total_milliseconds // 1000
    seconds = total_seconds % 60
    total_minutes = total_seconds // 60
    minutes = total_minutes % 60
    hours = total_minutes // 60

    return f"{hours:02}:{minutes:02}:{seconds:02},{milliseconds:03}"

In [10]:
def create_srt_file(segments, output_filename):
    """Creates a standard SubRip (SRT) subtitle file."""
    print(f"3. Generating SRT file: {output_filename}")
    with open(output_filename, 'w', encoding='utf-8') as f:
        for i, chunk in enumerate(segments, 1):
            # Check for None timestamp before subscripting
            if chunk.get('timestamp') is None or len(chunk['timestamp']) < 2:
                print(f"Warning: Segment {i} missing timestamp data. Skipping segment.")
                continue

            start_time = format_time(chunk['timestamp'][0])
            end_time = format_time(chunk['timestamp'][1])
            text = chunk['text'].strip()

            # SRT format:
            # 1
            # 00:00:00,000 --> 00:00:05,000
            # Hello world
            f.write(str(i) + '\n')
            f.write(f"{start_time} --> {end_time}\n")
            f.write(text + '\n\n')

    print(f"   -> SRT file saved successfully!")

In [11]:
def extract_audio(video_path, audio_output_path):
    """Extracts the audio track from a video file using moviepy."""
    print(f"1. Extracting audio from {video_path}...")
    try:
        video = VideoFileClip(video_path)
        video.audio.write_audiofile(audio_output_path, codec='mp3')
        print(f"   -> Audio successfully extracted to {audio_output_path}")
        video.close()
        return audio_output_path
    except Exception as e:
        print(f"Error during audio extraction: {e}")
        return None

## Main Processing

### Step 1: Upload video

In [12]:
video_file = SAMPLE_VIDEO_FILEPATH
audio_file = SAMPLE_AUDIO_FILEPATH

if IN_COLAB:
    try:
        # Import Colab-specific file utilities
        from google.colab import files
        print("Detected Google Colab environment. Displaying file upload widget...")

        # Display the upload dialog
        uploaded = files.upload()

        # Process uploaded files (this part is optional, but good for demo)
        if uploaded:
            print("\nUpload complete! Files uploaded:")
            for filename, content in uploaded.items():
                print(f"- {filename} ({len(content)} bytes)")
                with open(filename, 'wb') as f:
                    f.write(content)
                video_file = "/content/" + filename
                audio_file = "/content/" + SAMPLE_AUDIO_FILEPATH
        else:
            print("No files were uploaded.")

    except ImportError:
        # Should not happen if 'google.colab' is in sys.modules,
        # but good practice to handle.
        print("Error: Running in a Colab-like environment, but 'files' module failed to import.")

Detected Google Colab environment. Displaying file upload widget...


Saving sample_video.mp4 to sample_video.mp4

Upload complete! Files uploaded:
- sample_video.mp4 (15961391 bytes)


### Step 2: Extract audio from video file

In [13]:
extracted_audio_file = extract_audio(video_file, audio_file)

1. Extracting audio from /content/sample_video.mp4...
MoviePy - Writing audio in /content/extracted_audio.mp3


MoviePy - Done.
   -> Audio successfully extracted to /content/extracted_audio.mp3


### Step 3: Initialize Pipeline for transcription

In [14]:
# Handle the HuggingFace Token
# HF_TOKEN handling: prefer environment or .env, otherwise prompt
load_dotenv()
hf = os.environ.get("HF_TOKEN")

if not hf:
    # Colab: ask for HF token (hidden) and persist to .env
    if IN_COLAB:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token:
            os.environ["HF_TOKEN"] = token
            # persist token for future cells (writes .env in workspace)
            with open(".env", "a") as envf:
                envf.write(f"\nHF_TOKEN={token}\n")
            print("HF_TOKEN set for session and appended to .env")
    else:
        # Local (VS Code): missing so prompt for it
        print("HF_TOKEN not found. Create a .env file with HF_TOKEN=your_token or paste it now.")
        token = getpass("Paste HF_TOKEN (hidden): ")
        if token:
            os.environ["HF_TOKEN"] = token
            with open(".env", "a") as envf:
                envf.write(f"\nHF_TOKEN={token}\n")
            print("HF_TOKEN written to .env")

HF_TOKEN set for session and appended to .env


In [15]:
pipe = pipeline(
    "automatic-speech-recognition",
    model=ASR_MODEL_NAME,
    device=device,
    chunk_length_s=30,  # chunk size in seconds
    stride_length_s=5,  # optional overlap between chunks
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/290M [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

preprocessor_config.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


### Step 4: Start transcription of audio file with pipeline

In [16]:
result = pipe(extracted_audio_file, return_timestamps=True)

Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.


In [17]:
# peek at result
result

{'text': 'イラッシュアイマゼいらっしゃいませ。すみません。かさを忘れてしまいました。かさですか。何時ごろでしたかあ、さっきです30分前ぐらいどんなカサですかビニールガサですビニールガサですか名前は書いてありますか?名前は書いてありませんあ!でもキティちゃんのストラップがついてますちょっと待ってくださいねはいお客様、これですか?それです!よかったー。名前、書いてくれませんか?え。あー、名前ですね。10書もおねがいします分かりましたあのー本人確認できるものって、ありますか?え、いや、本人ですけど!マイナンバーカードは、運転面教証は?あの、カサですよ。ただのビニールガッサー。すみません。ルールですので。スタッフェインスタッフェインスタッフェインあのーコピーをとってもいいですか?もういりません!カッ、いりません!イマイズミ先生、お疲れ様です。イマイズミ先生、お疲れ様です。イマイズミ先生、お疲れ様です。イマイズミ先生、お疲れ様です。イマイズミ先生、お疲れ様です。イマイズミ先生、お疲れ様です。お疲れ様ですんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんんはい、これ、今月の漫画です。ありがとうございます。ちょっと見ますね。13144516ページはい 大丈夫です先生、お疲れ様でしたあ、ちょっと待ってください読んでくださいださいよあの先生 お願いがあり

### Step 5: Create subtitle file

In [18]:
create_srt_file(result['chunks'], "output_subtitles.srt")

3. Generating SRT file: output_subtitles.srt
   -> SRT file saved successfully!


## Credit
- Sample video (to cross-check transcription)
    - [Learn Japanese with Short Dramas - WAKU WAKU Japanese](https://www.youtube.com/watch?v=i2UEIUI3XRI)